# CD45 / CD3E / CD8a Intensity Analysis

This notebook explores marker intensity distributions for immune-related channels (CD45, CD3E, CD8a) in CellDIVE multiplexed imaging data.

**Sections:**
1. **Full-image histograms** — Intensity distribution across all pixels (from TIFF channels)
2. **Masked histograms** — Intensity distribution only inside segmented cells (excludes background)
3. **Napari viewer** — Interactive visualization of CellDIVE channels

Use the checkboxes to select which markers to overlay. Vertical lines show mean (solid), median (dotted), 95th and 99th percentiles (dashed) — useful for setting positive/negative thresholds.

## 1. Imports

Load required packages: tifffile, zarr, Plotly (interactive plots), ipywidgets (checkboxes), and Napari/OME-Zarr for visualization.

In [1]:
import tifffile                     # for reading TIFF files
import ipywidgets as widgets
from IPython.display import display, clear_output
import plotly.graph_objects as go  # interactive zoom/pan, no extra backend
import numpy as np                  # for array handling
import pandas as pd                 # for stats table
from pathlib import Path            # convenient path manipulation

import napari
import zarr
import dask.array as da
from ome_zarr.io import parse_url
from ome_zarr.reader import Reader

## 2. Full-Image Histogram

Intensity distribution across **all pixels** in each channel. Data is loaded from `../input/{channel}.tif` and subsampled (every 100th pixel) for speed.

- **CD45** — Pan-leukocyte marker
- **CD3E** — T cell marker
- **CD8a** — Cytotoxic T cell marker

Use the Plotly toolbar to zoom and pan. The stats table shows mean, p50, p95, p99 for threshold exploration.

In [2]:
# Configuration 
channels = ["CD45", "CD3E", "CD8a"]
marker_colors = {
    "CD45": "royalblue",
    "CD3E": "darkorange",
    "CD8a": "mediumseagreen",
}
# Create UI Elements (No observers attached manually)
# We use a dictionary for the checkboxes to map them directly to function arguments
checkbox_dict = {c: widgets.Checkbox(value=False, description=c) for c in channels}
ui_box = widgets.HBox(list(checkbox_dict.values()))

# Define the Plotting Function
# The arguments must match the keys in checkbox_dict
def plot_histogram(CD45, CD3E, CD8a):
    active_map = {"CD45": CD45, "CD3E": CD3E, "CD8a": CD8a}
    active_channels = [k for k, v in active_map.items() if v]

    if not active_channels:
        print("Select one or more channels to overlay their distributions.")
        return

    fig = go.Figure()
    
    percentiles = [50, 95, 99]
    line_dashes = {"mean": "solid", "p50": "dot", "p95": "dash", "p99": "dashdot"}
    stats_rows = []
    
    for chan in active_channels:
        path = Path(f'../input/{chan}.tif')
        if not path.exists():
            continue
            
        img = tifffile.memmap(path, mode='r')
        sample = img[::100, ::100].ravel()
        color = marker_colors.get(chan, "steelblue")
        
        mean_val = np.mean(sample)
        p50 = np.percentile(sample, 50)
        p95 = np.percentile(sample, 95)
        p99 = np.percentile(sample, 99)
        stats_rows.append({"Marker": chan, "mean": int(mean_val), "p50": int(p50), "p95": int(p95), "p99": int(p99)})
        
        counts, bin_edges = np.histogram(sample, bins=256, range=(0, 65535))
        fig.add_trace(go.Scatter(
            x=bin_edges, y=np.concatenate([[0], counts]),
            mode='lines', line=dict(color=color, width=2), name=chan, line_shape='hv'
        ))
        
        fig.add_vline(x=mean_val, line_color=color, line_dash="solid", line_width=1.5, opacity=0.8)
        for p in percentiles:
            pval = np.percentile(sample, p)
            fig.add_vline(x=pval, line_color=color, line_dash=line_dashes[f"p{p}"], line_width=1, opacity=0.6)

    if stats_rows:
        stats_df = pd.DataFrame(stats_rows).set_index("Marker")
        display(stats_df)

    fig.update_layout(
        title="Overlaid Marker Intensity Distributions (Log Scale)",
        xaxis_title="16-bit Intensity Value",
        yaxis_title="Pixel Count (Log)",
        xaxis=dict(range=[0, 65535]),
        yaxis=dict(type="log"),
        showlegend=True,
        legend=dict(yanchor="top", y=1, xanchor="right", x=1),
        height=450,
        margin=dict(t=60),
        annotations=[dict(
            text="Lines: — mean  ··· p50  - - p95  -·- p99",
            xref="paper", yref="paper", x=0.02, y=0.98, showarrow=False,
            xanchor="left", yanchor="top", font=dict(size=10),
            bgcolor="rgba(245,222,179,0.6)"
        )]
    )
    display(fig)

# Use interactive_output to link UI and Function
# This manages the threading and prevents the "triple-fire" issue
out = widgets.interactive_output(plot_histogram, checkbox_dict)

# Display everything
display(ui_box, out)

Output()

## 3. Masked Histogram (Inside Cells Only)

Intensity distribution for pixels **inside segmented cells** only. Background (mask = 0) is excluded.

- **Mask:** `cellpose_masks_dapi_only_9tiles.zarr` — Cellpose segmentation on DAPI channel
- **Source:** CellDIVE zarr channels (CD45, CD3E, CD8a)
- **Subsampling:** Every 100th pixel for memory efficiency

This view is more relevant for cell-level thresholding because it excludes autofluorescence and background. Compare with the full-image histogram above to see the effect of masking.

To use a different mask, change `MASK_ZARR` to another path (e.g. `cellpose_masks_dapi+panck_9tiles.zarr`).

In [3]:
# Histogram of marker intensities INSIDE cell mask (DAPI-only Cellpose segmentation)
# Uses only pixels where mask > 0 (inside segmented cells) — excludes background

MASK_ZARR = "../output/cellpose_output/cellpose_masks_dapi_only_9tiles.zarr"
CELLDIVE_ZARR = "../data/CellDIVE_SLIDE-045.zarr"
# Channel index in CellDIVE zarr: CD45=1, CD3E=2, CD8a=4
CHANNEL_MAP = {"CD45": 1, "CD3E": 2, "CD8a": 4}
SUBSAMPLE_STEP = 100  # every 100th pixel for memory/speed

mask_store = zarr.open(MASK_ZARR, mode="r")
celldive_store = zarr.open(CELLDIVE_ZARR, mode="r")
mask_level0 = mask_store["0"]
celldive_level0 = celldive_store["0"]

def plot_histogram_masked(CD45, CD3E, CD8a):
    active_map = {"CD45": CD45, "CD3E": CD3E, "CD8a": CD8a}
    active_channels = [k for k, v in active_map.items() if v]
    if not active_channels:
        print("Select one or more channels.")
        return

    fig = go.Figure()
    percentiles = [50, 95, 99]
    line_dashes = {"mean": "solid", "p50": "dot", "p95": "dash", "p99": "dashdot"}
    stats_rows = []

    mask_sub = np.array(mask_level0[::SUBSAMPLE_STEP, ::SUBSAMPLE_STEP])
    in_cell = mask_sub > 0

    for chan in active_channels:
        idx = CHANNEL_MAP[chan]
        ch_sub = np.array(celldive_level0[idx, ::SUBSAMPLE_STEP, ::SUBSAMPLE_STEP])
        intensities = ch_sub[in_cell].ravel()

        if len(intensities) == 0:
            print(f"No pixels in cells for {chan}")
            continue

        color = marker_colors.get(chan, "steelblue")
        mean_val = np.mean(intensities)
        p50 = np.percentile(intensities, 50)
        p95 = np.percentile(intensities, 95)
        p99 = np.percentile(intensities, 99)
        stats_rows.append({"Marker": chan, "mean": int(mean_val), "p50": int(p50), "p95": int(p95), "p99": int(p99)})

        counts, bin_edges = np.histogram(intensities, bins=256, range=(0, 65535))
        fig.add_trace(go.Scatter(
            x=bin_edges, y=np.concatenate([[0], counts]),
            mode='lines', line=dict(color=color, width=2), name=chan, line_shape='hv'
        ))
        fig.add_vline(x=mean_val, line_color=color, line_dash="solid", line_width=1.5, opacity=0.8)
        for p in percentiles:
            pval = np.percentile(intensities, p)
            fig.add_vline(x=pval, line_color=color, line_dash=line_dashes[f"p{p}"], line_width=1, opacity=0.6)

    if stats_rows:
        display(pd.DataFrame(stats_rows).set_index("Marker"))

    fig.update_layout(
        title="Marker Intensities INSIDE Cells (DAPI-only mask) — Log Scale",
        xaxis_title="16-bit Intensity Value",
        yaxis_title="Pixel Count (Log)",
        xaxis=dict(range=[0, 65535]),
        yaxis=dict(type="log"),
        showlegend=True,
        legend=dict(yanchor="top", y=1, xanchor="right", x=1),
        height=450,
        margin=dict(t=60),
        annotations=[dict(
            text=f"Mask: cellpose_masks_dapi_only_9tiles · step={SUBSAMPLE_STEP}",
            xref="paper", yref="paper", x=0.02, y=0.98, showarrow=False,
            xanchor="left", yanchor="top", font=dict(size=10),
            bgcolor="rgba(245,222,179,0.6)"
        )]
    )
    display(fig)

# Uses same checkboxes as cell above — toggling updates both histograms
out_masked = widgets.interactive_output(plot_histogram_masked, checkbox_dict)
display(out_masked)

Output()

## 4. Napari Viewer

Loads the CellDIVE zarr and Cellpose masks using the same logic as `napari_load_raw+mask.py`:

- **Image:** All 23 channels in one layer with channel selector (use the slider/dropdown to switch channels)
- **Masks:** DAPI-only and DAPI+PanCK segmentation overlays
- **Scale:** 0.325 µm/px for correct spatial dimensions

**Note:** Opens Napari in a separate window. The cell blocks until the window is closed.

In [4]:
# Load CellDIVE zarr + masks (same logic as napari_load_raw+mask.py)
import json

# Paths: try project root first, then relative to notebooks/
for base in [Path.cwd(), Path.cwd().parent]:
    p = base / "data" / "CellDIVE_SLIDE-045.zarr"
    if p.exists():
        store_path = str(p)
        break
else:
    store_path = "../data/CellDIVE_SLIDE-045.zarr"

# 1. Load CellDIVE multiscale pyramid
group = zarr.open(store_path, mode="r")
pyramid = [da.from_zarr(group[str(i)]) for i in range(5)]
n_channels = pyramid[0].shape[0]

# 2. Channel names and colors from .zattrs
zattrs_path = Path(store_path) / ".zattrs"
names, colors = [], []
if zattrs_path.exists():
    with open(zattrs_path) as f:
        meta = json.load(f)
        channels_meta = meta.get("omero", {}).get("channels", [])
        if not channels_meta and "multiscales" in meta:
            channels_meta = meta["multiscales"][0].get("omero", {}).get("channels", [])
for i in range(n_channels):
    if i < len(channels_meta):
        names.append(channels_meta[i].get("label", f"Ch{i}"))
        c = channels_meta[i].get("color", "FFFFFF")
        colors.append(f"#{c}" if not c.startswith("#") else c)
    else:
        names.append(f"Ch{i}")
        colors.append("gray")

# 3. Create viewer and add image (single layer, channel_axis=0)
viewer = napari.Viewer()
viewer.add_image(
    pyramid,
    channel_axis=0,
    name=names,
    colormap=colors,
    multiscale=True,
    blending="additive",
    contrast_limits=[[0, 3000]] * n_channels,
    scale=(0.325, 0.325),
)

# 4. Load masks (from napari_load_raw+mask.py)
def _prefer_zarr(p):
    p = Path(p)
    if p.suffix.lower() in {".tif", ".tiff"} and p.with_suffix(".zarr").is_dir():
        return str(p.with_suffix(".zarr"))
    return str(p)

def _load_mask(viewer, path, name, color_rgba, contour=1):
    path = Path(_prefer_zarr(path))
    if not path.exists():
        print(f"Mask not found: {path}")
        return
    if path.suffix.lower() == ".zarr":
        mg = zarr.open(str(path), mode="r")
        keys = sorted([k for k in mg.keys() if str(k).isdigit()], key=lambda x: int(x))
        data = [da.from_zarr(mg[k]) for k in keys]
        multiscale = True
    else:
        mm = tifffile.memmap(str(path))
        if mm.ndim > 2:
            mm = mm.squeeze()
        data = da.from_array(mm, chunks=(1024, 1024))
        multiscale = False
    layer = viewer.add_labels(data, name=name, multiscale=multiscale, opacity=1.0, scale=(0.325, 0.325))
    layer.contour = contour
    try:
        from napari.utils import DirectLabelColormap
        layer.colormap = DirectLabelColormap(color_dict={None: color_rgba})
    except Exception:
        pass

# Mask paths (relative to project root or notebooks/)
mask_dir = Path(store_path).resolve().parent.parent / "output" / "cellpose_output"
if not mask_dir.exists():
    mask_dir = Path("../output/cellpose_output").resolve()
MASK_CONFIGS = [
    (mask_dir / "cellpose_masks_dapi_only_9tiles.zarr", "DAPI only", [1.0, 1.0, 1.0, 0.9]),
    (mask_dir / "cellpose_masks_dapi+panck_9tiles.zarr", "DAPI + PanCK", [0.75, 1.0, 0.75, 0.9]),
]
for path, name, color in MASK_CONFIGS:
    _load_mask(viewer, str(path), name, color)

napari.run()